# Fetch external datasets (run in Google Colab)

Downloads the road network, traffic volumes (AADT), county boundaries, main roads and NOAA weather for FDOT District 2 into `MyDrive/Data/external`.

Run the cells top to bottom. Each dataset is fetched in its own cell, so if one public server is down, re-run just that cell later.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/Data/external'
!mkdir -p {OUT}

In [ ]:
# Get the latest project code
!rm -rf /content/repo
!git clone -q https://github.com/pascalantwi/accident-project-with-sando /content/repo
!pip install -q osmnx

## 1. Road network and candidate sites (OpenStreetMap)
Motorways (interstates, expressways) in District 2, plus interchanges, rest areas and service plazas. About 5–15 minutes.

In [ ]:
!python /content/repo/src/preprocessing/fetch_external_data.py --outdir {OUT} --only osm_interstates osm_candidate_sites

## 2. Traffic volumes (FDOT AADT)
Current year and the last five years, District 2 only.

In [ ]:
!python /content/repo/src/preprocessing/fetch_external_data.py --outdir {OUT} --only aadt_current aadt_historical

## 3. County boundaries and main roads (US Census TIGER)

In [ ]:
!python /content/repo/src/preprocessing/fetch_external_data.py --outdir {OUT} --only fl_counties fl_primary_secondary_roads

## 4. Hourly precipitation (NOAA)
All NOAA weather stations in District 2 active in 2022–2024.

In [ ]:
!python /content/repo/src/preprocessing/fetch_external_data.py --outdir {OUT} --only noaa_weather

## 5. Check what was downloaded
`sources.json` records where and when each dataset came from (cite it in the paper).

In [ ]:
!ls -lh {OUT}
!cat {OUT}/sources.json

## 6. Bundle for upload to GitHub
GitHub's website accepts files up to 25 MB. This zips each dataset separately; upload the zips to `data/external/` in the repo (Add file → Upload files), or tell Claude where they are on Drive.

In [ ]:
import os, zipfile
os.makedirs(f'{OUT}/zips', exist_ok=True)
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if os.path.isfile(p) and not f.endswith('.zip'):
        z = os.path.join(OUT, 'zips', f + '.zip')
        with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
            zf.write(p, f)
        print(f'{f:40s} {os.path.getsize(z)/1e6:7.1f} MB zipped')